In [17]:
import os
import wandb
import string
import random
import urllib.request
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

In [19]:
# Setting seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# logging in to Weights & Biases
wandb.login()

True

In [20]:
# downloading the Letter Recognition dataset from UCI
urllib.request.urlretrieve(
    "https://archive.ics.uci.edu/ml/machine-learning-databases/letter-recognition/letter-recognition.data",
    "letter-recognition.data"
)

# mapping each letter A-Z to a number from 0 to 25
letter_to_id = {ch: i for i, ch in enumerate(string.ascii_uppercase)}
# storing features and labels
X_list, y_list = [], []

# reading each row in the dataset
with open("letter-recognition.data", "r") as f:
    for line in f:
        parts = line.strip().split(",")
        y_list.append(letter_to_id[parts[0]])
        X_list.append([int(v) for v in parts[1:]])

# converting lists into NumPy arrays
X = np.array(X_list, dtype=np.float32)
y = np.array(y_list, dtype=np.int32)
print(f"Loaded {X.shape[0]} samples, {X.shape[1]} features, {len(np.unique(y))} classes")

# splitting the data into train, validation, and test sets
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=SEED
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.176, stratify=y_temp, random_state=SEED
)
# displaying the split sizes
print(f"Train: {len(y_train)} | Val: {len(y_val)} | Test: {len(y_test)}")

# creating XGBoost data matrices
xg_train = xgb.DMatrix(X_train, label=y_train)
xg_val   = xgb.DMatrix(X_val,   label=y_val)
xg_test  = xgb.DMatrix(X_test,  label=y_test)

# starting the weights and biases run
run = wandb.init(project="Lab5", name="xgboost_letter_recognition")

# giving the model hyperparameters
params = {
    "objective":        "multi:softprob",
    "eta":              0.08,
    "max_depth":        8,
    "min_child_weight": 3,
    "subsample":        0.8,
    "colsample_bytree": 0.8,
    "reg_lambda":       1.5,
    "reg_alpha":        0.3,
    "eval_metric":      "mlogloss",
    "num_class":        26,
    "seed":             SEED,
    "nthread":          4,
    "verbosity":        0,
}
# logging hyperparameters to weight and biases
wandb.config.update(params)

# using training and validation sets for training
watchlist = [(xg_train, "train"), (xg_val, "val")]

# training the XGBoost model
bst = xgb.train(
    params,
    xg_train,
    num_boost_round=300,
    evals=watchlist,
    early_stopping_rounds=20,
    verbose_eval=25,
    callbacks=[wandb.xgboost.WandbCallback()],
)
# displaying the best training results
print(f"\nBest iteration: {bst.best_iteration}")
print(f"Best validation mlogloss: {bst.best_score}")

run.summary["best_iteration"] = int(bst.best_iteration)
run.summary["best_score"] = float(bst.best_score)

# predicting probabilities on the test set
prob = bst.predict(xg_test)
pred = np.argmax(prob, axis=1)

# computing evaluation metrics
accuracy   = accuracy_score(y_test, pred)
precision  = precision_score(y_test, pred, average="weighted", zero_division=0)
recall     = recall_score(y_test, pred, average="weighted", zero_division=0)
f1         = f1_score(y_test, pred, average="weighted", zero_division=0)
error_rate = 1 - accuracy

print(f"Accuracy   = {accuracy:.4f}")
print(f"Precision  = {precision:.4f}")
print(f"Recall     = {recall:.4f}")
print(f"F1         = {f1:.4f}")
print(f"Error Rate = {error_rate:.4f}")

run.summary["Accuracy"]   = accuracy
run.summary["Precision"]  = precision
run.summary["Recall"]     = recall
run.summary["F1"]         = f1
run.summary["Error Rate"] = error_rate

# classification report
class_names = list(string.ascii_uppercase)
print("\n" + classification_report(y_test, pred, target_names=class_names, zero_division=0))

# creating a feature importance table
feature_names = [
    "x-box", "y-box", "width", "high", "onpix", "x-bar", "y-bar", "x2bar",
    "y2bar", "xybar", "x2ybr", "xy2br", "x-ege", "xegvy", "y-ege", "yegvx"
]

# getting the feature importance scores from the model
importance = bst.get_score(importance_type="gain")
imp_table = wandb.Table(columns=["feature", "gain"])
for i, fname in enumerate(feature_names):
    imp_table.add_data(fname, importance.get(f"f{i}", 0.0))

wandb.log({
    "feature_importance": wandb.plot.bar(
        imp_table, "feature", "gain", title="Feature Importance"
    )
})
# Confusion matrix
wandb.sklearn.plot_confusion_matrix(y_test, pred, labels=list(range(26)))

# saving the trained model
os.makedirs("artifacts", exist_ok=True)
bst.save_model("artifacts/xgb_letter_model.json")

# creating and logging the model artifact to weight and biases
art = wandb.Artifact("letter-recognition-xgb", type="model",
                      description=f"XGBoost 26-class letter recognition (acc {accuracy:.4f})")
art.add_file("artifacts/xgb_letter_model.json")
run.log_artifact(art)

run.finish()

Loaded 20000 samples, 16 features, 26 classes
Train: 14008 | Val: 2992 | Test: 3000


[0]	train-mlogloss:2.76541	val-mlogloss:2.76862
[25]	train-mlogloss:0.62290	val-mlogloss:0.70943
[50]	train-mlogloss:0.27683	val-mlogloss:0.37787
[75]	train-mlogloss:0.16096	val-mlogloss:0.26485
[100]	train-mlogloss:0.10741	val-mlogloss:0.21108
[125]	train-mlogloss:0.07851	val-mlogloss:0.18187
[150]	train-mlogloss:0.06140	val-mlogloss:0.16474
[175]	train-mlogloss:0.05070	val-mlogloss:0.15380
[200]	train-mlogloss:0.04346	val-mlogloss:0.14601
[225]	train-mlogloss:0.03823	val-mlogloss:0.14062
[250]	train-mlogloss:0.03433	val-mlogloss:0.13634
[275]	train-mlogloss:0.03141	val-mlogloss:0.13324
[299]	train-mlogloss:0.02916	val-mlogloss:0.13097

Best iteration: 299
Best validation mlogloss: 0.13097377808357766
Accuracy   = 0.9610
Precision  = 0.9615
Recall     = 0.9610
F1         = 0.9611
Error Rate = 0.0390

              precision    recall  f1-score   support

           A       0.99      0.98      0.99       118
           B       0.92      0.96      0.94       115
           C       0.96 

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


best_iteration,▁
best_score,▁
epoch,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▃▃▃▄▅▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇████
train-mlogloss,█▅▄▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val-mlogloss,█▄▃▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
Accuracy,0.961
Error Rate,0.039
F1,0.96106
Precision,0.96148
Recall,0.961
best_iteration,299
